In [100]:
import pandas as pd, numpy as np
from pathlib import Path
from datetime import datetime
from unidecode import unidecode
import sys, os, traceback

src = Path.cwd().parent/"src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
pd.set_option('display.max_columns', None)

In [101]:
from routes.Paths import(
    Ruta_Vtex,
    Ruta_global,
    Ruta_local
)
from utils.functions import load_files_from_folder
from utils.Data_ModeloComercial.b_processing_crm_ventas import process_crm_ventas

In [102]:
required_columns = [
    "order",
     "sequence",
     "creation date",
     "client document",
     "client name",
     "client last name",
     "quantity_sku",
     "total value",
     "shipping value",
     "sku total price",
     "sla type",
     "courrier",
     "estimate delivery date",
     "payment value"
]
df_vtex = load_files_from_folder(
    Ruta_Vtex,
    set(required_columns),
    required_columns
)

columns_vtex = [
    "order",
    "sequence",
    "creation date",
    "client document",
    "client name",
    "client last name",
    "quantity_sku",
    "total value",
    "shipping value",
    "sku total price",
    "sla type",
    "courrier",
    "estimate delivery date",
    "payment value",
    "source_file",
    "file_date"
]

df_vtex = df_vtex[columns_vtex]

In [103]:
date_time_columns = ["creation date", "estimate delivery date", "file_date"]
for col in date_time_columns:
    df_vtex[col] = pd.to_datetime(df_vtex[col],
                                  format="mixed",
                                  utc=True,
                                  errors="coerce").dt.tz_localize(None)
df_vtex["Total_Venta"] = (df_vtex["shipping value"] + df_vtex["sku total price"]).round(2)
df_vtex["date"] = df_vtex["creation date"].dt.date
df_vtex = df_vtex.sort_values(by=["client document", "creation date"], ascending=[True, False]).reset_index(drop=True)
df_vtex["client document"] = df_vtex["client document"].astype(str)

In [104]:
df_crm = process_crm_ventas(save_file = False)
columnas_crm_requeridas = [
    "NÚMERO_DOCUMENTO_DEL_CLIENTE",
    "RADICADO",
    "IdTipificacion",
    "NIVEL_1",
    "NIVEL_2",
    "NIVEL_3",
    "TIPO_DE_LLAMADA",
    "NOMBRE_CLIENTE",
    "CONTACTO_DE_CLIENTE",
    "NOMBRE_ALIADO",
    "Id_opl",
    "NOMBRE_ASESOR",
    "Correo_contratista",
    "FECHA_DE_INICIO",
    "START_DATE",
    "FECHA_FIN",
    "END_DATE",
    "START_MONTH_DATE",
    "OFRECIMIENTO_EFECTIVO",
    "Aplica_Ofrecimiento",
    "VALOR_PUNTOS",
    "CANAL_DE_ATENCIÓN",
    "Colas",
    "SKILL",
    "IdMasterSkill",
    "ESTADO_GESTIÓN",
    "NÚMERO_PEDIDO"
]

df_crm["FECHA_DE_INICIO"] = pd.to_datetime(
    df_crm["FECHA_DE_INICIO"],
    errors="coerce"
)

df_crm["FECHA_FIN"] = pd.to_datetime(
    df_crm["FECHA_FIN"],
    errors="coerce"
)

df_crm["Prioridad"] = (
    df_crm["OFRECIMIENTO_EFECTIVO"]
        .map({"Si": 1, "No": 2})
)

df_crm = df_crm.sort_values(
    by=[
        "NÚMERO_DOCUMENTO_DEL_CLIENTE",
        "START_DATE",
        "Prioridad"
    ]
)
df_crm["NÚMERO_DOCUMENTO_DEL_CLIENTE"] = df_crm["NÚMERO_DOCUMENTO_DEL_CLIENTE"].astype(str)

In [105]:
df_vtex = df_vtex.merge(
    df_crm[columnas_crm_requeridas],
        how="left",
        left_on=[
            "client document",
            "date"
        ],
        right_on=[
            "NÚMERO_DOCUMENTO_DEL_CLIENTE",
            "START_DATE"
        ]
    ).drop(columns=["NÚMERO_DOCUMENTO_DEL_CLIENTE","START_DATE"])

df_vtex["HORAS_DESDE_INICIO"] = ((df_vtex["creation date"] - df_vtex["FECHA_DE_INICIO"]).dt.total_seconds()/3600).round(2)
df_vtex["HORAS_DESDE_FIN"] = ((df_vtex["creation date"] - df_vtex["FECHA_FIN"]).dt.total_seconds()/3600).round(2)

df_vtex["VALIDACION_VENTA"] = np.where(
    (df_vtex["OFRECIMIENTO_EFECTIVO"].eq("Si")) & 
    (df_vtex["HORAS_DESDE_INICIO"].between(0, 24) | df_vtex["HORAS_DESDE_FIN"].between(0, 24)),
    "Venta efectiva",
    "Venta no efectiva"
)

In [106]:
df_vtex["VALIDACION_VENTA"].value_counts()

VALIDACION_VENTA
Venta no efectiva    304029
Venta efectiva         3032
Name: count, dtype: int64

In [107]:
df_vtex.to_csv(os.path.join(Ruta_local, "12.Data_Vtex_Ventas.csv"), index=False)